# Week 6: Advanced Monte Carlo simulation, distribution fitting
ENGI 1151 Computational Tools for Engineers and Scientists

Run each cell with **Shift + Enter**, from top to bottom: later cells use variables from earlier ones. The full explanations are on the week 6 notes page.

## 2. Maximum likelihood by hand
The log-likelihood of the normal distribution is largest at the sample mean and standard deviation.

In [ ]:
import numpy as np

x = np.array([4.1, 5.0, 5.6, 6.2, 6.4, 7.1, 7.8, 9.0])

def log_likelihood(mu, sigma):
    pdf = np.exp(-(x - mu)**2 / (2 * sigma**2)) / (np.sqrt(2 * np.pi) * sigma)
    return np.sum(np.log(pdf))

mu_hat, sigma_hat = np.mean(x), np.std(x)
print("MLE: mu = %.3f, sigma = %.3f, log-likelihood = %.3f" % (mu_hat, sigma_hat, log_likelihood(mu_hat, sigma_hat)))
for mu, sigma in [(6.0, 1.46), (6.4, 2.0), (5.0, 1.0)]:
    print("mu = %.2f, sigma = %.2f: log-likelihood = %.3f" % (mu, sigma, log_likelihood(mu, sigma)))

## 3. Common distributions in `scipy.stats`

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

examples = [("norm", stats.norm(loc=0, scale=1.5), -6, 6),
            ("t (df=5)", stats.t(df=5), -6, 6),
            ("uniform", stats.uniform(loc=-3, scale=6), -6, 6),
            ("triang", stats.triang(c=0.3, loc=1, scale=7), 0, 12),
            ("expon", stats.expon(scale=1.5), 0, 12),
            ("lognorm", stats.lognorm(s=0.5, scale=3), 0, 12),
            ("gamma", stats.gamma(a=2, scale=1.2), 0, 12),
            ("weibull_min", stats.weibull_min(c=2, scale=4), 0, 12),
            ("beta (a=5, b=2)", stats.beta(a=5, b=2), 0, 1)]

fig, axes = plt.subplots(3, 3, figsize=(9, 6.5))
for ax, (name, dist, lo, hi) in zip(axes.flat, examples):
    x = np.linspace(lo, hi, 400)
    ax.fill_between(x, dist.pdf(x), color="#E8D9EA")
    ax.plot(x, dist.pdf(x), color="#68246D", lw=2)
    ax.set_title("%s   skewness %.2f" % (name, dist.stats(moments="s")), fontsize=10)
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## 5. Fitting and testing in Python
### 5.1 Fitting
200 pump repair times (hours), simulated so that everyone has the same data.

In [ ]:
import numpy as np
import scipy.stats as stats

np.random.seed(7)
data = np.random.gamma(2.5, 4, 200)          # 200 repair times (h)

normal_params = stats.norm.fit(data)
expo_params = stats.expon.fit(data, floc=0)
lognorm_params = stats.lognorm.fit(data, floc=0)
gamma_params = stats.gamma.fit(data, floc=0)

print("normal (loc, scale):        ", np.round(normal_params, 3))
print("exponential (loc, scale):   ", np.round(expo_params, 3))
print("lognormal (s, loc, scale):  ", np.round(lognorm_params, 3))
print("gamma (a, loc, scale):      ", np.round(gamma_params, 3))

g = stats.gamma(*gamma_params)               # frozen distribution
print("gamma mean %.2f h, P(time > 40 h) = %.5f" % (g.mean(), g.sf(40)))

### 5.2 Fitted pdfs over the histogram

In [ ]:
import matplotlib.pyplot as plt

candidates = {"norm": {}, "expon": {"floc": 0},
              "lognorm": {"floc": 0}, "gamma": {"floc": 0}}

x = np.linspace(0, data.max(), 300)
plt.hist(data, bins=25, density=True, color="skyblue", edgecolor="black", alpha=0.7)
for name, options in candidates.items():
    dist = getattr(stats, name)              # e.g. stats.gamma
    params = dist.fit(data, **options)
    plt.plot(x, dist.pdf(x, *params), lw=2, label=name)
plt.xlabel("Repair time (h)")
plt.ylabel("Density")
plt.legend()
plt.show()

### 5.3 CDFs and the Kolmogorov–Smirnov test

In [ ]:
import matplotlib.pyplot as plt

xs = np.sort(data)
ecdf = np.arange(1, len(xs) + 1) / len(xs)   # empirical CDF at each sorted point
plt.step(xs, ecdf, where="post", color="black", lw=1.5, label="data (ECDF)")

x = np.linspace(0, data.max(), 300)
print("dist        D       p   worst at x")
for name, options in {"norm": {}, "expon": {"floc": 0},
                      "lognorm": {"floc": 0}, "gamma": {"floc": 0}}.items():
    dist = getattr(stats, name)
    params = dist.fit(data, **options)
    result = stats.kstest(data, dist(*params).cdf)
    print("%-8s %6.3f %7.4f %8.2f" % (name, result.statistic, result.pvalue, result.statistic_location))
    plt.plot(x, dist.cdf(x, *params), lw=1.8, label=name)
plt.xlabel("Repair time (h)")
plt.ylabel("Cumulative probability")
plt.legend()
plt.show()

## 6. Percentiles and probabilities from the CDF

In [ ]:
import matplotlib.pyplot as plt

g = stats.gamma(*stats.gamma.fit(data, floc=0))

print("median:          %.2f h" % g.ppf(0.5))
print("95th percentile: %.2f h   (from the data: %.2f h)" % (g.ppf(0.95), np.percentile(data, 95)))
print("P(time <= 5 h) = %.3f" % g.cdf(5))
print("P(time > 24 h) = %.4f" % g.sf(24))

x = np.linspace(0, 40, 300)
plt.plot(x, g.cdf(x), color="#68246D", lw=2.5)
p95 = g.ppf(0.95)
plt.plot([0, p95, p95], [0.95, 0.95, 0], "r--")          # probability -> value
plt.plot([5, 5, 0], [0, g.cdf(5), g.cdf(5)], "b--")       # value -> probability
plt.text(p95 + 0.5, 0.05, "ppf(0.95) = %.1f h" % p95, color="red")
plt.text(5.5, g.cdf(5) - 0.07, "cdf(5) = %.3f" % g.cdf(5), color="blue")
plt.xlabel("Repair time x (h)")
plt.ylabel("F(x)")
plt.grid(True)
plt.show()

## 7. The wind farm, continued
The week 5 simulation of the daily energy of the wind farm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def turbine_power(v, theta, eta, rho=1.225, D=80):
    A = np.pi * D**2 / 4
    return 0.5 * rho * A * eta * (v * np.cos(theta))**3 / 1e6   # MW

ab = np.array([[5, 2], [6, 2], [4, 1.5], [5, 1.5], [7, 2],
               [6, 3], [5, 2.5], [8, 3], [4, 2], [6, 2]])
np.random.seed(1)
N = 10_000                                     # simulated days
v = np.random.normal(8, 1.5, (N, 1))           # m/s
theta = np.random.normal(0, 0.26, (N, 1))      # rad
eta = 0.62 + 0.33 * np.random.beta(ab[:, 0], ab[:, 1], (N, 10))
energy = 24 * turbine_power(v, theta, eta).sum(axis=1)   # MWh/day

print("mean %.0f MWh, empirical P95 %.0f MWh" % (energy.mean(), np.percentile(energy, 5)))
plt.hist(energy, bins=60, density=True, color="skyblue", edgecolor="black")
plt.xlabel("Energy in one day (MWh)")
plt.ylabel("Density")
plt.show()

### Exercise 7.1
1. Fit `norm`, `lognorm`, `gamma` and `weibull_min` to `energy` (use `floc=0` for the last three).
2. Plot the density histogram with the four fitted pdfs.
3. Run the K–S test for each candidate: D, p-value and the location of the largest gap. Which pass at α = 0.05? Which is best?
4. With the best fit, find P95 with `ppf(0.05)` and compare it with `np.percentile(energy, 5)`.
5. Find the probability of a day with less than 100 MWh, and with more than 600 MWh.
6. Repeat with N = 1 000 days. Does the decision change? Why?

In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt

# Write your code here


## 8. Extra: sensitivity analysis
Correlation of each input with the daily energy, shown as a tornado diagram.

In [ ]:
import matplotlib.pyplot as plt

inputs = {"wind speed v": v[:, 0],
          "wind angle |θ|": np.abs(theta[:, 0]),
          "mean efficiency": eta.mean(axis=1),
          "turbine 1 efficiency": eta[:, 0]}

corr = {name: np.corrcoef(values, energy)[0, 1] for name, values in inputs.items()}
for name, r in corr.items():
    print("%-22s r = %+.3f" % (name, r))

order = sorted(corr, key=lambda k: abs(corr[k]))       # largest bar on top
plt.barh(order, [corr[k] for k in order], color=["#68246D" if corr[k] > 0 else "#B3261E" for k in order])
plt.axvline(0, color="black", lw=1)
plt.xlabel("Correlation with the daily energy")
plt.title("Sensitivity of the daily energy to the inputs")
plt.tight_layout()
plt.show()

## Exercises
1. Fit a `beta` distribution to the efficiencies of turbine 1, `eta[:, 0]`. Do you recover a = 5, b = 2? (Hint: `stats.beta.fit(eta[:, 0], floc=0.62, fscale=0.33)`.)
2. Fit the normal distribution to the repair times by hand with `np.mean` and `np.std`, and compare with `stats.norm.fit(data)`.
3. How does the K–S p-value of the normal fit to the wind farm energy change with N = 50, 200 and 1 000 days?
4. Make scatter plots of each input (`v[:, 0]`, `np.abs(theta[:, 0])`, `eta.mean(axis=1)`) against `energy`. Which relationships are not linear?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


In [ ]:
# Exercise 4
